## Introduction

Dans le projet précédent, nous avons construit progressivement un assistant intelligent pour l’Hôtel Le Belvédère.

L’objectif était de comprendre comment transformer un modèle de langage généraliste en assistant capable de répondre à partir de données réelles.

Nous avons commencé avec un modèle de langage seul :

In [ ]:
llm.invoke("Je me nomme Diabaté Youssouf")

Puis nous avons constaté une première limite : le modèle ne conserve pas automatiquement toutes les informations d’une requête à l’autre.

Nous avons ensuite ajouté plusieurs mécanismes.

1. Le modèle de langage

Le modèle de langage permet de comprendre une question et de produire une réponse.

Cependant, il ne connaît pas nécessairement les informations spécifiques de notre application.

Par exemple, si nous lui demandons :

Quel est le prix d’une chambre Classic à l’Hôtel Le Belvédère ?

le modèle peut produire une réponse plausible sans réellement connaître le prix officiel de l’hôtel.

Cela peut provoquer une hallucination.

Une hallucination correspond ici à une information générée par le modèle qui paraît correcte mais qui n’est pas fondée sur une source fiable.

2. Le RAG

Pour résoudre ce problème, nous avons ajouté une base documentaire contenant les informations officielles de l’hôtel.

Le fonctionnement était alors :

```Question du client
        ↓
Recherche dans les documents
        ↓
Documents pertinents
        ↓
Contexte
        ↓
Modèle de langage
        ↓
Réponse

Le RAG permet donc au modèle de répondre à partir d'informations récupérées dans les documents.

Nous avons notamment utilisé :

In [ ]:
moteur_recherche.invoke(question)

pour récupérer les passages pertinents.

Le modèle ne devait alors plus simplement « deviner » la réponse.

3. La mémoire conversationnelle

Nous avons ensuite ajouté une mémoire de conversation.

L'objectif était de conserver les échanges précédents.

Par exemple :
```
Client : Je m'appelle Youssouf.

Assistant : Enchanté Youssouf.

Client : Quel type de chambre me conseillez-vous ?

La deuxième question peut être interprétée avec le contexte de la conversation précédente.

Mais il faut faire attention à une distinction importante :

La mémoire conserve le contexte conversationnel, mais elle ne garantit pas que les informations générées par le modèle soient vraies.

La mémoire et le RAG répondent donc à deux problèmes différents.

4. Les outils

Nous avons également créé des fonctions capables d'effectuer des opérations précises.

Par exemple :
```
def tool_calculer_facture(nb_nuits, prix_par_nuit):
    taxe = 2.50 * nb_nuits
    total = (nb_nuits * prix_par_nuit) + taxe

    return total

Cette fonction effectue réellement un calcul.

Le modèle de langage n'a donc pas besoin d'inventer le résultat.

Nous avons également créé un outil permettant de vérifier la disponibilité d'une chambre.

Nous avions donc plusieurs composants :
```
LLM
│
├── RAG
│
├── Calcul de facture
│
├── Vérification de disponibilité
│
└── Mémoire

**Le problème rencontré**

À ce stade, notre assistant possède plusieurs capacités.

Il peut :

comprendre une question avec le LLM ;
rechercher des informations avec le RAG ;
calculer une facture ;
vérifier une disponibilité ;
conserver une partie de la conversation.

Mais une nouvelle question apparaît :

Qui décide quel composant doit être utilisé ?

Prenons trois exemples.

Exemple 1
`Quel est le prix d'une chambre Classic ?`

Il faut utiliser le RAG.

Exemple 2
`Je reste 4 nuits à 110 € par nuit.`
`Combien vais-je payer ?`

Il faut utiliser l'outil de calcul.

Exemple 3
`Reste-t-il une Suite Panoramic ?`

Il faut utiliser l'outil de disponibilité.

Nous avions donc besoin d'un **mécanisme** capable d'organiser **automatiquement** le **traitement**.

### Pourquoi LangGraph ?

C'est précisément à ce niveau que LangGraph devient intéressant.

LangGraph permet de représenter le fonctionnement d'un agent sous la forme d'un graphe d'exécution.

Un graphe est composé principalement de :

State : les informations transportées pendant l'exécution ;
Nodes : les étapes de traitement ;
Edges : les connexions entre les étapes ;
Conditional Edges : les connexions dont la destination dépend d'une décision.

On peut représenter notre futur agent ainsi :
```
                    Question
                       │
                       ▼
                 ┌───────────┐
                 │ Analyse   │
                 │   LLM     │
                 └─────┬─────┘
                       │
          ┌────────────┼────────────┐
          │            │            │
          ▼            ▼            ▼
        RAG          Calcul       Dispo
          │            │            │
          └────────────┼────────────┘
                       │
                       ▼
                 Réponse finale

L'intérêt de LangGraph est donc de transformer notre système de fonctions indépendantes en workflow contrôlé.

## **Objectif du nouveau projet**

Dans ce projet, nous allons reprendre progressivement les éléments du projet précédent et les organiser avec LangGraph.

Nous allons commencer avec un exemple extrêmement simple.

Nous n'allons pas immédiatement construire l'agent hôtelier complet.

Nous allons d'abord comprendre le fonctionnement de LangGraph avec de petits exemples.

La progression sera :
```
1. Comprendre le State
        ↓
2. Créer un Node
        ↓
3. Relier deux Nodes
        ↓
4. Utiliser START et END
        ↓
5. Ajouter une décision
        ↓
6. Ajouter des Conditional Edges
        ↓
7. Utiliser Mistral dans un Node
        ↓
8. Connecter notre RAG
        ↓
9. Connecter nos outils
        ↓
10. Ajouter la mémoire
        ↓
11. Construire l'agent complet
        ↓
12. Tester et analyser ses limites

1. Installation de LangGraph
Explication

Avant de construire notre premier graphe, nous devons installer la bibliothèque.

LangGraph est une bibliothèque Python conçue pour construire des applications dans lesquelles plusieurs étapes doivent être exécutées dans un ordre précis et parfois selon des décisions.

Dans notre cas, elle va nous permettre de gérer le fonctionnement de notre agent.

### Installation

Dans le terminal du projet :

`uv add langgraph`

Une fois l'installation terminée, nous pouvons importer les éléments nécessaires.

In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END

### Explication des imports
`TypedDict`
`from typing import TypedDict`

`TypedDict` nous permettra de définir la structure de notre State.

Nous verrons précisément ce concept dans la prochaine partie.

**`StateGraph`**

`from langgraph.graph import StateGraph`

StateGraph est l'objet que nous utiliserons pour construire notre graphe.

Il permettra notamment d'ajouter :

`Nodes`
`Edges`
`Conditional Edges`
`START`
`START`

`START` représente le point de départ du graphe.

`END`
`END`

`END` représente la fin de l'exécution.

Nous pouvons donc déjà imaginer :
```
START
  ↓
Node 1
  ↓
Node 2
  ↓
END


2. Comprendre le State
### Explication

Le State est l'un des concepts les plus importants de LangGraph.

Un agent doit constamment manipuler des informations.

Dans notre projet, il devra par exemple connaître :

la question du client
la décision prise
les documents récupérés
le résultat d'un outil
la réponse finale

Toutes ces informations doivent pouvoir être transmises d'une étape à une autre.

C'est précisément le rôle du State.

On peut voir le State comme une fiche contenant les informations courantes de l'exécution.

Par exemple :
```
┌──────────────────────────────┐
│          STATE               │
├──────────────────────────────┤
│ question = "Bonjour"         │
│ decision = ""                │
│ resultat = ""                │
│ reponse = ""                 │
└──────────────────────────────┘

Un Node peut lire cette fiche.

Il peut également modifier certaines informations.

2. Comprendre le State
### Explication

Le State est l'un des concepts les plus importants de LangGraph.

Un agent doit constamment manipuler des informations.

Dans notre projet, il devra par exemple connaître :
```
la question du client
la décision prise
les documents récupérés
le résultat d'un outil
la réponse finale

Toutes ces informations doivent pouvoir être transmises d'une étape à une autre.

C'est précisément le rôle du State.

On peut voir le State comme une fiche contenant les informations courantes de l'exécution.

Par exemple :
```
┌──────────────────────────────┐
│          STATE               │
├──────────────────────────────┤
│ question = "Bonjour"         │
│ decision = ""                │
│ resultat = ""                │
│ reponse = ""                 │
└──────────────────────────────┘

Un Node peut lire cette fiche.

Il peut également modifier certaines informations.

Exemple très simple

Nous allons créer un State minimal.

In [ ]:
class AgentState(TypedDict):
    question: str
    reponse: str

Nous avons maintenant défini un State contenant deux informations :
```
question
reponse

`question` sera une chaîne de caractères.

`reponse` sera également une chaîne de caractères.

**Pourquoi utiliser TypedDict ?**

Cette ligne :

`class AgentState(TypedDict):`

indique la structure attendue de notre State.

Elle nous permet de dire :

« **Mon agent va manipuler un état qui possède ces informations.** »

Par exemple :

```
{
    "question": "Bonjour",
    "reponse": "Bonjour, comment puis-je vous aider ?"
}

3. Premier Node
### Explication

Un Node représente une étape de traitement.

Un Node est généralement une fonction Python.

Par exemple :

In [ ]:
def saluer(state):
    return {
        "reponse": "Bonjour !"
    }

Cette fonction reçoit le State.

Elle peut donc accéder aux informations présentes dans le State.

Elle retourne ensuite les informations qu'elle souhaite modifier.


Application

Créons un deuxième exemple.

In [ ]:
def analyser_question(state):
    question = state["question"]

    return {
        "reponse": f"Vous avez écrit : {question}"
    }

La fonction récupère :

In [ ]:
question = state["question"]

Puis elle construit une réponse.

Si le State contient :
```
{
    "question": "Bonjour",
    "reponse": ""
}


le Node peut produire :
```
{
    "reponse": "Vous avez écrit : Bonjour"
}

4. Construire notre premier graphe

Nous avons maintenant :

* un State ;
* une fonction ;
* un futur Node.

Nous pouvons construire le graphe.

In [ ]:
workflow = StateGraph(AgentState)

Cette instruction signifie :

Nous allons construire un graphe utilisant le `State AgentState`.

### Ajouter le Node

In [ ]:
workflow.add_node("analyse", analyser_question)

Nous donnons au Node le nom : `analyse`

et nous lui associons la fonction : `analyser_question`

Nous avons maintenant :
```
analyse
   │
   ▼
analyser_question()

5. Ajouter les connexions

Un Node seul ne suffit pas.

Nous devons indiquer au graphe :

« **Quand l'exécution commence, va vers le Node analyse.** »

Nous utilisons :

In [ ]:
workflow.add_edge(START, "analyse")

Puis nous devons indiquer que le Node termine le traitement :

In [ ]:
workflow.add_edge("analyse", END)

Notre graphe devient :
```
START
  │
  ▼
analyse
  │
  ▼
 END

6. Compiler le graphe

Le graphe est maintenant défini.

Mais il doit être compilé avant d'être exécuté.

In [ ]:
app = workflow.compile()

Nous pouvons maintenant utiliser `app`.

7. Exécuter le graphe

Nous allons envoyer une question au graphe.

In [ ]:
resultat = app.invoke({
    "question": "Bonjour",
    "reponse": ""
})

Puis afficher le résultat :

In [ ]:
print(resultat)

Nous obtenons une structure similaire à :

```
{
    "question": "Bonjour",
    "reponse": "Vous avez écrit : Bonjour"
}

### Observation

Nous venons de construire notre premier workflow LangGraph.

Il est extrêmement simple :
```
Question
   │
   ▼
START
   │
   ▼
analyse
   │
   ▼
END

Le point important est que nous avons maintenant séparé :
```
Les données
     ↓
   State

Le traitement
     ↓
   Node

L'ordre d'exécution
     ↓
   Edges

Cette séparation va devenir très importante lorsque notre agent possédera plusieurs capacités.

### À toi de jouer

Modifie le Node afin qu'il transforme la question en majuscules.

Par exemple :

Entrée :
`"bonjour"`

Sortie :
`"BONJOUR"`

Le State doit rester identique.

8. Pourquoi avons-nous besoin de plusieurs Nodes ?

Dans notre exemple, un seul Node suffit.

Mais notre agent hôtelier possède plusieurs traitements.

Nous pouvons donc imaginer :
```
                    Question
                       │
                       ▼
                    Analyse
                       │
             ┌─────────┼─────────┐
             ▼         ▼         ▼
            RAG      Calcul     Dispo
             │         │         │
             └─────────┼─────────┘
                       ▼
                    Réponse

Chaque opération devient un Node indépendant.

Cela présente un avantage important :

Chaque étape possède une responsabilité clairement définie.

Par exemple :

In [ ]:
def node_rag(state):
    ...

ne sera responsable que de la recherche documentaire.

Et :

In [ ]:
def node_calcul(state):
    ...

sera responsable du calcul.

9. Les Conditional Edges

Jusqu'ici, notre graphe suivait toujours le même chemin :

`START → analyse → traitement → END`

Mais un agent doit pouvoir prendre des décisions.

Par exemple :
```
Question
   ↓
Analyse
   ↓
Quelle opération ?
   │
   ├── RAG
   │
   ├── CALCUL
   │
   └── DISPO

C'est ici que les Conditional Edges interviennent.

Un Conditional Edge permet de dire :

« En fonction de la valeur obtenue, choisis le prochain Node. »

Exemple

Notre State peut maintenant contenir :

In [ ]:
class AgentState(TypedDict):
    question: str
    decision: str
    reponse: str

Le Node d'analyse peut produire :

```
{
    "decision": "RAG"
}

Nous avons ensuite une fonction de routage :

In [ ]:
def choisir_direction(state):
    return state["decision"]

Cette fonction lit simplement la décision.

Si :

In [ ]:
state["decision"] == "RAG"

elle retourne : `RAG`

Le graphe peut alors diriger l'exécution vers le Node correspondant.

10. Application à notre agent hôtelier

Nous pouvons maintenant imaginer le graphe final :
```
                         START
                           │
                           ▼
                    Analyse question
                           │
              ┌────────────┼────────────┐
              │            │            │
              ▼            ▼            ▼
             RAG         CALCUL        DISPO
              │            │            │
              └────────────┼────────────┘
                           │
                           ▼
                    Réponse finale
                           │
                           ▼
                          END

C'est cette architecture que nous allons progressivement construire.

Le projet précédent nous avait permis de construire les différentes briques :
```
LLM
RAG
Mémoire
Tools

Le nouveau projet avec LangGraph va maintenant répondre à une autre question :

Comment organiser toutes ces briques dans un workflow intelligent, lisible et contrôlable ?

C'est précisément l'intérêt de LangGraph.